In [ ]:
def add(P, Q):
    A = P._parent
    assert Q._parent == P._parent

    A0, A1, A2, A3, A4, A5, A6, A7, A8 = P.coordinates()
    B0, B1, B2, B3, B4, B5, B6, B7, B8 = Q.coordinates()

    X0 = A4*A8*B0^2 - A3*A6*B1*B2 - A1*A2*B3*B6 + A0^2*B4*B8
    X1 = -A5*A8*B0*B1 + A3*A7*B2^2 + A2^2*B3*B7 - A0*A1*B5*B8
    X2 = A3*A8*B1^2 - A4*A7*B0*B2 - A0*A2*B4*B7 + A1^2*B3*B8
    X3 = -A7*A8*B0*B3 + A6^2*B1*B5 + A1*A5*B6^2 - A0*A3*B7*B8
    X4 = A8^2*B0*B4 - A6*A7*B2*B5 - A2*A5*B6*B7 + A0*A4*B8^2
    X5 = -A6*A8*B1*B4 + A7^2*B0*B5 + A0*A5*B7^2 - A1*A4*B6*B8
    X6 = A1*A8*B3^2 - A0*A6*B4*B5 - A4*A5*B0*B6 + A3^2*B1*B8
    X7 = -A2*A8*B3*B4 + A0*A7*B5^2 + A5^2*B0*B7 - A3*A4*B2*B8
    X8 = A0*A8*B4^2 - A1*A7*B3*B5 - A3*A5*B1*B7 + A4^2*B0*B8

    return A([X0, X1, X2, X3, X4, X5, X6, X7, X8])

In [ ]:
####################################
####################################
## trying addition formulas

from itertools import product
from isogeny_chain_dim2 import *

k = 4
p = 8*3^k - 1
F1 = GF(p)
R.<x> = F1[]
Fp.<om> = GF(p^2, modulus=x^2+x+1)
omega = om

while True:
    E1 = EllipticCurve(Fp, [1,0])
    # "random" isogenous curve with the same product structure
    #P = E1.lift_x(2)
    P = 3^5*E1.random_element()
    E2 = E1.isogeny(P).codomain()
    
    # symplectic 3^k-torsion basis
    P1,P2,Q1,Q2 = create_basis(E1,E2,k,omega=omega)
    
    # random kernel for a (3^k,3^k)-isogeny
    a = ZZ.random_element(3^(k-1))
    b = ZZ.random_element(3^(k-2))
    c = ZZ.random_element(3^(k-1))
    b = 1 + 3*b # need b!=0, so that the first isogeny is non-diagonal.
    
    # (3^k,3^k)- group on (E1 x E2) in Hessian form (+ auxiliary information)
    # where the group is <(P1 + a*Q1, b*Q2),(b*Q1, P2 + c*Q2)
    (R,S),(R_9,S_9) = translate_to_Hessian((P1,P2,Q1,Q2),k,(a,b,c),E1,E2)
    
    A = R._parent
    
    Phi = compute_isogeny_chain((R,S), (R_9,S_9), k-1, (a,b,c))
    
    # we can push points lothrough the isogeny
    H2,H1 = A._elliptic_curves
    Rand1 = E1.random_element()
    Rand2 = E2.random_element()
    R1 = H1.map_point(Rand1)
    R2 = H2.map_point(Rand2)
    R12 = A([R2,R1]);
    phi_R12 = Phi(R12)
    
    # implicit test (note that addition on the Hessian is not implemented)
    # R12 + first kernel generator
    Test1 = Rand1 + 3*(P1 + a*Q1)
    Test2 = Rand2 + 3*b*Q2
    T1 = H1.map_point(Test1)
    T2 = H2.map_point(Test2)
    T12 = A([T2,T1])
    phi_T12 = Phi(T12)
    
    # R12 + second kernel generator
    Test1 = Rand1 + 3*(b*Q1)
    Test2 = Rand2 + 3*(P2 + c*Q2)
    S1 = H1.map_point(Test1)
    S2 = H2.map_point(Test2)
    S12 = A([S2,S1])
    phi_S12 = Phi(S12)
    
    if (phi_R12 == phi_S12 and phi_R12 == phi_T12):
        break

def random_points():
    Rand1 = E1.random_element()
    Rand2 = E2.random_element()
    R1 = H1(Rand1)
    R2 = H2(Rand2)
    R12 = A([R2,R1])
    phi_R12 = Phi(R12)
    return (Rand1, Rand2, phi_R12)

# def get_sample():
#     Rand1 = E1.random_element()
#     Rand2 = E2.random_element()
#     R1 = H1(Rand1)
#     R2 = H2(Rand2)
#     R12 = A([R2,R1])
    
#     Tand1 = E1.random_element()
#     Tand2 = E2.random_element()
#     T1 = H1(Tand1)
#     T2 = H2(Tand2)
#     T12 = A([T2,T1])
    
#     RT1 = H1(Rand1 + Tand1)
#     RT2 = H2(Rand2 + Tand2)
    
#     RT12 = A([RT2,RT1])
    
#     R12 = A([R2,R1])
#     T12 = A([T2,T1])

#     return R12, T12, RT12

def get_sample():
    while True:
        R1, R2, R12 = random_points()
        T1, T2, T12 = random_points()
        RT1 = R1 + T1
        RT2 = R2 + T2
        RT1 = H1(RT1)
        RT2 = H2(RT2)
        RT12 = A([RT2,RT1])
        phi_RT12 = Phi(RT12)
    
        if prod(R12)*prod(T12)*prod(phi_RT12) != 0:
            break
    
    return R12, T12, phi_RT12

def index_to_mons(index):
    i0 = index%3
    i1 = index//3
    mons = [(c00+3*c01,c10+3*c11)
            for c00 in range(3)
            for c01 in range(3)
            for c10 in range(3)
            for c11 in range(3)
            if (c00 + c10)%3 == i0
            and (c01 + c11)%3 == i1
            and 3*c11+c10 > 3*c01+c00]
    return mons
    
def square_mons(mons):
    mons = [mon1+mon2 for mon1 in mons for mon2 in mons]
    return mons

def get_monomials():
    monss = []
    for x in range(9):
        mons = index_to_mons(x)
        mons = square_mons(mons)
        monss.append(mons)
    return monss

monss = get_monomials()
ls = [len(mons) for mons in monss]

def zero_list(length):
    return [0 for _ in range(length)]

def get_rows(sample):
    R = sample[0]
    T = sample[1]
    RT = sample[2]
    rows = []
    for x in range(1,9):
        betweenzeros = sum([ls[it] for it in range(x-1)])
        postzeros = sum([ls[it] for it in range(x+1,9)])
            
        row = ([RT[x]*R[i]*R[j]*T[m]*T[n] for (i,j,m,n) in monss[0]]
               + zero_list(betweenzeros)
               + [-RT[0]*R[i]*R[j]*T[m]*T[n] for (i,j,m,n) in monss[x]]
               + zero_list(postzeros)
              )

        rows.append(row)

    return rows
    
M = []
for s in range(20):
    sample = get_sample()
    rows = get_rows(sample)
    M += rows
    
M = matrix(Fp, M)

b0 = M.right_kernel().basis()[0]

B = Phi.codomain()

hs = B._h
ds = B._d

hs,ds,b0[0:16]

In [ ]:
import ast

with open("basis.txt") as f:
    data = ast.literal_eval(f.read())

In [ ]:
[1 for i in data[1124] if i != 0]

In [ ]:
from isogeny_chain_dim2 import *

def get_meta_sample():
    ####################################
    ####################################
    ## trying addition formulas
    
    from itertools import product
    
    k = 4
    p = 8*3^k - 1
    F1 = GF(p)
    R.<x> = F1[]
    Fp.<om> = GF(p^2, modulus=x^2+x+1)
    omega = om
    
    while True:
        E1 = EllipticCurve(Fp, [1,0])
        # "random" isogenous curve with the same product structure
        #P = E1.lift_x(2)
        P = 3^5*E1.random_element()
        E2 = E1.isogeny(P).codomain()
        
        # symplectic 3^k-torsion basis
        P1,P2,Q1,Q2 = create_basis(E1,E2,k,omega=omega)
        
        # random kernel for a (3^k,3^k)-isogeny
        a = ZZ.random_element(3^(k-1))
        b = ZZ.random_element(3^(k-2))
        c = ZZ.random_element(3^(k-1))
        b = 1 + 3*b # need b!=0, so that the first isogeny is non-diagonal.
        
        # (3^k,3^k)- group on (E1 x E2) in Hessian form (+ auxiliary information)
        # where the group is <(P1 + a*Q1, b*Q2),(b*Q1, P2 + c*Q2)
        (R,S),(R_9,S_9) = translate_to_Hessian((P1,P2,Q1,Q2),k,(a,b,c),E1,E2)
        
        A = R._parent
        
        Phi = compute_isogeny_chain((R,S), (R_9,S_9), k-1, (a,b,c))
        
        # we can push points lothrough the isogeny
        H2,H1 = A._elliptic_curves
        Rand1 = E1.random_element()
        Rand2 = E2.random_element()
        R1 = H1.map_point(Rand1)
        R2 = H2.map_point(Rand2)
        R12 = A([R2,R1]);
        phi_R12 = Phi(R12)
        
        # implicit test (note that addition on the Hessian is not implemented)
        # R12 + first kernel generator
        Test1 = Rand1 + 3*(P1 + a*Q1)
        Test2 = Rand2 + 3*b*Q2
        T1 = H1.map_point(Test1)
        T2 = H2.map_point(Test2)
        T12 = A([T2,T1])
        phi_T12 = Phi(T12)
        
        # R12 + second kernel generator
        Test1 = Rand1 + 3*(b*Q1)
        Test2 = Rand2 + 3*(P2 + c*Q2)
        S1 = H1.map_point(Test1)
        S2 = H2.map_point(Test2)
        S12 = A([S2,S1])
        phi_S12 = Phi(S12)
        
        if (phi_R12 == phi_S12 and phi_R12 == phi_T12):
            break
    
    def random_points():
        Rand1 = E1.random_element()
        Rand2 = E2.random_element()
        R1 = H1(Rand1)
        R2 = H2(Rand2)
        R12 = A([R2,R1])
        phi_R12 = Phi(R12)
        return (Rand1, Rand2, phi_R12)
    
    # def get_sample():
    #     Rand1 = E1.random_element()
    #     Rand2 = E2.random_element()
    #     R1 = H1(Rand1)
    #     R2 = H2(Rand2)
    #     R12 = A([R2,R1])
        
    #     Tand1 = E1.random_element()
    #     Tand2 = E2.random_element()
    #     T1 = H1(Tand1)
    #     T2 = H2(Tand2)
    #     T12 = A([T2,T1])
        
    #     RT1 = H1(Rand1 + Tand1)
    #     RT2 = H2(Rand2 + Tand2)
        
    #     RT12 = A([RT2,RT1])
        
    #     R12 = A([R2,R1])
    #     T12 = A([T2,T1])
    
    #     return R12, T12, RT12
    
    def get_sample():
        while True:
            R1, R2, R12 = random_points()
            T1, T2, T12 = random_points()
            RT1 = R1 + T1
            RT2 = R2 + T2
            RT1 = H1(RT1)
            RT2 = H2(RT2)
            RT12 = A([RT2,RT1])
            phi_RT12 = Phi(RT12)
        
            if prod(R12)*prod(T12)*prod(phi_RT12) != 0:
                break
        
        return R12, T12, phi_RT12
    
    def index_to_mons(index):
        i0 = index%3
        i1 = index//3
        mons = [(c00+3*c01,c10+3*c11)
                for c00 in range(3)
                for c01 in range(3)
                for c10 in range(3)
                for c11 in range(3)
                if (c00 + c10)%3 == i0
                and (c01 + c11)%3 == i1
                and 3*c11+c10 > 3*c01+c00]
        return mons
        
    def square_mons(mons):
        mons = [mon1+mon2 for mon1 in mons for mon2 in mons]
        return mons
    
    def get_monomials():
        monss = []
        for x in range(9):
            mons = index_to_mons(x)
            mons = square_mons(mons)
            monss.append(mons)
        return monss
    
    monss = get_monomials()
    ls = [len(mons) for mons in monss]
    
    def zero_list(length):
        return [0 for _ in range(length)]
    
    def get_rows(sample):
        R = sample[0]
        T = sample[1]
        RT = sample[2]
        rows = []
        for x in range(1,9):
            betweenzeros = sum([ls[it] for it in range(x-1)])
            postzeros = sum([ls[it] for it in range(x+1,9)])
                
            row = ([RT[x]*R[i]*R[j]*T[m]*T[n] for (i,j,m,n) in monss[0]]
                   + zero_list(betweenzeros)
                   + [-RT[0]*R[i]*R[j]*T[m]*T[n] for (i,j,m,n) in monss[x]]
                   + zero_list(postzeros)
                  )
    
            rows.append(row)
    
        return rows
        
    M = []
    for s in range(20):
        sample = get_sample()
        rows = get_rows(sample)
        M += rows
        
    M = matrix(Fp, M)
    
    b0 = M.right_kernel().basis()[0]
    
    B = Phi.codomain()
    
    hs = B._h
    ds = B._d
    
    return hs,ds,b0[0:17]

ls = len([1 for i1 in range(5) for j1 in range(i1,5) for i2 in range(5) for j2 in range(i2,5)])

print(ls)

def zero_list(length):
    return [0 for _ in range(length)]

def get_rows(sample):
    hs = sample[0]
    ds = sample[1]
    cs = sample[2]
    rows = []
    
    for x in range(1,17):
        betweenzeros = sum([ls for it in range(x-1)])
        postzeros = sum([ls for it in range(x+1,17)])
            
        row = ([cs[x]*hs[i1]*hs[j1]*ds[i2]*ds[j2] for i1 in range(5) for j1 in range(i1,5) for i2 in range(4) for j2 in range(i2,4)]
               + zero_list(betweenzeros)
               + [-cs[0]*hs[i1]*hs[j1]*ds[i2]*ds[j2] for i1 in range(5) for j1 in range(i1,5) for i2 in range(4) for j2 in range(i2,4)]
               + zero_list(postzeros)
              )

        rows.append(row)

    return rows

M = []
for s in range(1):
    try:
        sample = get_meta_sample()
        rows = get_rows(sample)
        M += rows
    except:
        True
    
M = matrix(Fp, M)

M

In [2]:
# script.sage

from isogeny_chain_dim2 import *

from itertools import product

k = 4
p = 8*3^k - 1
F1 = GF(p)
R.<x> = F1[]
Fp.<om> = GF(p^2, modulus=x^2+x+1)
omega = om


def get_meta_sample():

    # We first sample a random abelian surface as the image of a (3,3)-isogeny chain
    # coming from E1 x E2, where E2 is isogenous to E1
    
    while True:
        E1 = EllipticCurve(Fp, [1,0])
        # "random" isogenous curve with the same product structure
        #P = E1.lift_x(2)
        P = 3^5*E1.random_element()
        E2 = E1.isogeny(P).codomain()
        
        # symplectic 3^k-torsion basis
        P1,P2,Q1,Q2 = create_basis(E1,E2,k,omega=omega)
        
        # random kernel for a (3^k,3^k)-isogeny
        a = ZZ.random_element(3^(k-1))
        b = ZZ.random_element(3^(k-2))
        c = ZZ.random_element(3^(k-1))
        b = 1 + 3*b # need b!=0, so that the first isogeny is non-diagonal.
        
        # (3^k,3^k)- group on (E1 x E2) in Hessian form (+ auxiliary information)
        # where the group is <(P1 + a*Q1, b*Q2),(b*Q1, P2 + c*Q2)
        (R,S),(R_9,S_9) = translate_to_Hessian((P1,P2,Q1,Q2),k,(a,b,c),E1,E2)
        
        A = R._parent
        
        Phi = compute_isogeny_chain((R,S), (R_9,S_9), k-1, (a,b,c))
        
        # we can push points lothrough the isogeny
        H2,H1 = A._elliptic_curves
        Rand1 = E1.random_element()
        Rand2 = E2.random_element()
        R1 = H1.map_point(Rand1)
        R2 = H2.map_point(Rand2)
        R12 = A([R2,R1]);
        phi_R12 = Phi(R12)
        
        # implicit test (note that addition on the Hessian is not implemented)
        # R12 + first kernel generator
        Test1 = Rand1 + 3*(P1 + a*Q1)
        Test2 = Rand2 + 3*b*Q2
        T1 = H1.map_point(Test1)
        T2 = H2.map_point(Test2)
        T12 = A([T2,T1])
        phi_T12 = Phi(T12)
        
        # R12 + second kernel generator
        Test1 = Rand1 + 3*(b*Q1)
        Test2 = Rand2 + 3*(P2 + c*Q2)
        S1 = H1.map_point(Test1)
        S2 = H2.map_point(Test2)
        S12 = A([S2,S1])
        phi_S12 = Phi(S12)
        
        if (phi_R12 == phi_S12 and phi_R12 == phi_T12):
            break

    # We now interpolate the addition formulae on our randomly generated surface
    
    def random_points():
        Rand1 = E1.random_element()
        Rand2 = E2.random_element()
        R1 = H1(Rand1)
        R2 = H2(Rand2)
        R12 = A([R2,R1])
        phi_R12 = Phi(R12)
        return (Rand1, Rand2, phi_R12)
    
    def get_sample():
        while True:
            R1, R2, R12 = random_points()
            T1, T2, T12 = random_points()
            RT1 = R1 + T1
            RT2 = R2 + T2
            RT1 = H1(RT1)
            RT2 = H2(RT2)
            RT12 = A([RT2,RT1])
            phi_RT12 = Phi(RT12)
        
            if prod(R12)*prod(T12)*prod(phi_RT12) != 0:
                break
        
        return R12, T12, phi_RT12
    
    def index_to_mons(index):
        i0 = index%3
        i1 = index//3
        mons = [(c00+3*c01,c10+3*c11)
                for c00 in range(3)
                for c01 in range(3)
                for c10 in range(3)
                for c11 in range(3)
                if (c00 + c10)%3 == i0
                and (c01 + c11)%3 == i1
                and 3*c11+c10 > 3*c01+c00]
        return mons
        
    def square_mons(mons):
        mons = [mon1+mon2 for mon1 in mons for mon2 in mons]
        return mons
    
    def get_monomials():
        monss = []
        for x in range(9):
            mons = index_to_mons(x)
            mons = square_mons(mons)
            monss.append(mons)
        return monss
    
    monss = get_monomials()
    ls = [len(mons) for mons in monss]
    
    def zero_list(length):
        return [0 for _ in range(length)]
    
    def get_rows(sample):
        R = sample[0]
        T = sample[1]
        RT = sample[2]
        rows = []
        for x in range(1,9):
            betweenzeros = sum([ls[it] for it in range(x-1)])
            postzeros = sum([ls[it] for it in range(x+1,9)])
                
            row = ([RT[x]*R[i]*R[j]*T[m]*T[n] for (i,j,m,n) in monss[0]]
                   + zero_list(betweenzeros)
                   + [-RT[0]*R[i]*R[j]*T[m]*T[n] for (i,j,m,n) in monss[x]]
                   + zero_list(postzeros)
                  )
    
            rows.append(row)
    
        return rows
        
    M = []
    for s in range(20):
        sample = get_sample()
        rows = get_rows(sample)
        M += rows
        
    M = matrix(Fp, M)

    K = M.right_kernel()
    
    b0 = K.basis()[0]
    
    B = Phi.codomain()
    
    hs = B._h
    ds = B._dfrom isogeny_chain_dim2 import *

from itertools import product

k = 4
p = 8*3^k - 1
F1 = GF(p)
R.<x> = F1[]
Fp.<om> = GF(p^2, modulus=x^2+x+1)
omega = om


def get_meta_sample():

    # We first sample a random abelian surface as the image of a (3,3)-isogeny chain
    # coming from E1 x E2, where E2 is isogenous to E1
    
    while True:
        E1 = EllipticCurve(Fp, [1,0])
        # "random" isogenous curve with the same product structure
        #P = E1.lift_x(2)
        P = 3^5*E1.random_element()
        E2 = E1.isogeny(P).codomain()
        
        # symplectic 3^k-torsion basis
        P1,P2,Q1,Q2 = create_basis(E1,E2,k,omega=omega)
        
        # random kernel for a (3^k,3^k)-isogeny
        a = ZZ.random_element(3^(k-1))
        b = ZZ.random_element(3^(k-2))
        c = ZZ.random_element(3^(k-1))
        b = 1 + 3*b # need b!=0, so that the first isogeny is non-diagonal.
        
        # (3^k,3^k)- group on (E1 x E2) in Hessian form (+ auxiliary information)
        # where the group is <(P1 + a*Q1, b*Q2),(b*Q1, P2 + c*Q2)
        (R,S),(R_9,S_9) = translate_to_Hessian((P1,P2,Q1,Q2),k,(a,b,c),E1,E2)
        
        A = R._parent
        
        Phi = compute_isogeny_chain((R,S), (R_9,S_9), k-1, (a,b,c))
        
        # we can push points lothrough the isogeny
        H2,H1 = A._elliptic_curves
        Rand1 = E1.random_element()
        Rand2 = E2.random_element()
        R1 = H1.map_point(Rand1)
        R2 = H2.map_point(Rand2)
        R12 = A([R2,R1]);
        phi_R12 = Phi(R12)
        
        # implicit test (note that addition on the Hessian is not implemented)
        # R12 + first kernel generator
        Test1 = Rand1 + 3*(P1 + a*Q1)
        Test2 = Rand2 + 3*b*Q2
        T1 = H1.map_point(Test1)
        T2 = H2.map_point(Test2)
        T12 = A([T2,T1])
        phi_T12 = Phi(T12)
        
        # R12 + second kernel generator
        Test1 = Rand1 + 3*(b*Q1)
        Test2 = Rand2 + 3*(P2 + c*Q2)
        S1 = H1.map_point(Test1)
        S2 = H2.map_point(Test2)
        S12 = A([S2,S1])
        phi_S12 = Phi(S12)
        
        if (phi_R12 == phi_S12 and phi_R12 == phi_T12):
            break

    # We now interpolate the addition formulae on our randomly generated surface
    
    def random_points():
        Rand1 = E1.random_element()
        Rand2 = E2.random_element()
        R1 = H1(Rand1)
        R2 = H2(Rand2)
        R12 = A([R2,R1])
        phi_R12 = Phi(R12)
        return (Rand1, Rand2, phi_R12)
    
    def get_sample():
        while True:
            R1, R2, R12 = random_points()
            T1, T2, T12 = random_points()
            RT1 = R1 + T1
            RT2 = R2 + T2
            RT1 = H1(RT1)
            RT2 = H2(RT2)
            RT12 = A([RT2,RT1])
            phi_RT12 = Phi(RT12)
        
            if prod(R12)*prod(T12)*prod(phi_RT12) != 0:
                break
        
        return R12, T12, phi_RT12
    
    def index_to_mons(index):
        i0 = index%3
        i1 = index//3
        mons = [(c00+3*c01,c10+3*c11)
                for c00 in range(3)
                for c01 in range(3)
                for c10 in range(3)
                for c11 in range(3)
                if (c00 + c10)%3 == i0
                and (c01 + c11)%3 == i1
                and 3*c11+c10 > 3*c01+c00]
        return mons
        
    def square_mons(mons):
        mons = [mon1+mon2 for mon1 in mons for mon2 in mons]
        return mons
    
    def get_monomials():
        monss = []
        for x in range(9):
            mons = index_to_mons(x)
            mons = square_mons(mons)
            monss.append(mons)
        return monss
    
    monss = get_monomials()
    ls = [len(mons) for mons in monss]
    
    def zero_list(length):
        return [0 for _ in range(length)]
    
    def get_rows(sample):
        R = sample[0]
        T = sample[1]
        RT = sample[2]
        rows = []
        for x in range(1,9):
            betweenzeros = sum([ls[it] for it in range(x-1)])
            postzeros = sum([ls[it] for it in range(x+1,9)])
                
            row = ([RT[x]*R[i]*R[j]*T[m]*T[n] for (i,j,m,n) in monss[0]]
                   + zero_list(betweenzeros)
                   + [-RT[0]*R[i]*R[j]*T[m]*T[n] for (i,j,m,n) in monss[x]]
                   + zero_list(postzeros)
                  )
    
            rows.append(row)
    
        return rows
        
    M = []
    for s in range(20):
        sample = get_sample()
        rows = get_rows(sample)
        M += rows
        
    M = matrix(Fp, M)

    K = M.right_kernel()
    
    b0 = K.basis()[0]
    
    B = Phi.codomain()
    
    hs = B._h
    ds = B._d
    
    return hs,ds,b0[0:17]

    
    return hs,ds,b0[0:17]

# We now interpolate global addition formulae, by interpolating over Abelian surfaces

# Precompute monomial indices
idxs = [
    (i1, j1, i2, j2)
    for i1 in range(5) for j1 in range(i1, 5)
    for i2 in range(4) for j2 in range(i2, 4)
]
ls = len(idxs)

def zero_list(length):
    return [0 for _ in range(length)]

def get_rows(sample):
    hs, ds, cs = sample
    
    rows = []
    
    for x in range(1,17):
        betweenzeros = sum([ls for it in range(x-1)])
        postzeros = sum([ls for it in range(x+1,17)])
            
        row = ([cs[x]*hs[i1]*hs[j1]*ds[i2]*ds[j2] for (i1, j1, i2, j2) in idxs]
               + zero_list(betweenzeros)
               + [-cs[0]*hs[i1]*hs[j1]*ds[i2]*ds[j2] for (i1, j1, i2, j2) in idxs]
               + zero_list(postzeros)
              )

        rows.append(row)

    return rows

# construct interpolation matrix
M = []
for s in range(1):
    try:
        sample = get_meta_sample()
        rows = get_rows(sample)
        M += rows
    except:
        True
    
M = matrix(Fp, M)

# compute kernel
K = M.right_kernel()

print(K)

bas = K.basis()[0]

with open('basis.txt', 'w') as op:
    op.write(str(bas))

Vector space of degree 2550 and dimension 2534 over Finite Field in om of size 647^2
Basis matrix:
2534 x 2550 dense matrix over Finite Field in om of size 647^2


In [5]:
# Replaced field Fp by F

from isogeny_chain_dim2 import *

from itertools import product

k = 4
p = 8*3^k - 1
F1 = GF(p)
R.<x> = F1[]
F.<om> = GF(p^2, modulus=x^2+x+1)
omega = om


def get_meta_sample():

    # We first sample a random abelian surface as the image of a (3,3)-isogeny chain
    # coming from E1 x E2, where E2 is isogenous to E1
    
    while True:
        E1 = EllipticCurve(Fp, [1,0])
        # "random" isogenous curve with the same product structure
        #P = E1.lift_x(2)
        P = 3^5*E1.random_element()
        E2 = E1.isogeny(P).codomain()
        
        # symplectic 3^k-torsion basis
        P1,P2,Q1,Q2 = create_basis(E1,E2,k,omega=omega)
        
        # random kernel for a (3^k,3^k)-isogeny
        a = ZZ.random_element(3^(k-1))
        b = ZZ.random_element(3^(k-2))
        c = ZZ.random_element(3^(k-1))
        b = 1 + 3*b # need b!=0, so that the first isogeny is non-diagonal.
        
        # (3^k,3^k)- group on (E1 x E2) in Hessian form (+ auxiliary information)
        # where the group is <(P1 + a*Q1, b*Q2),(b*Q1, P2 + c*Q2)
        (R,S),(R_9,S_9) = translate_to_Hessian((P1,P2,Q1,Q2),k,(a,b,c),E1,E2)
        
        A = R._parent
        
        Phi = compute_isogeny_chain((R,S), (R_9,S_9), k-1, (a,b,c))
        
        # we can push points lothrough the isogeny
        H2,H1 = A._elliptic_curves
        Rand1 = E1.random_element()
        Rand2 = E2.random_element()
        R1 = H1.map_point(Rand1)
        R2 = H2.map_point(Rand2)
        R12 = A([R2,R1]);
        phi_R12 = Phi(R12)
        
        # implicit test (note that addition on the Hessian is not implemented)
        # R12 + first kernel generator
        Test1 = Rand1 + 3*(P1 + a*Q1)
        Test2 = Rand2 + 3*b*Q2
        T1 = H1.map_point(Test1)
        T2 = H2.map_point(Test2)
        T12 = A([T2,T1])
        phi_T12 = Phi(T12)
        
        # R12 + second kernel generator
        Test1 = Rand1 + 3*(b*Q1)
        Test2 = Rand2 + 3*(P2 + c*Q2)
        S1 = H1.map_point(Test1)
        S2 = H2.map_point(Test2)
        S12 = A([S2,S1])
        phi_S12 = Phi(S12)
        
        if (phi_R12 == phi_S12 and phi_R12 == phi_T12):
            break

    # We now interpolate the addition formulae on our randomly generated surface
    
    def random_points():
        Rand1 = E1.random_element()
        Rand2 = E2.random_element()
        R1 = H1(Rand1)
        R2 = H2(Rand2)
        R12 = A([R2,R1])
        phi_R12 = Phi(R12)
        return (Rand1, Rand2, phi_R12)
    
    def get_sample():
        while True:
            R1, R2, R12 = random_points()
            T1, T2, T12 = random_points()
            RT1 = R1 + T1
            RT2 = R2 + T2
            RT1 = H1(RT1)
            RT2 = H2(RT2)
            RT12 = A([RT2,RT1])
            phi_RT12 = Phi(RT12)
        
            if prod(R12)*prod(T12)*prod(phi_RT12) != 0:
                break
        
        return R12, T12, phi_RT12
    
    def index_to_mons(index):
        i0 = index%3
        i1 = index//3
        mons = [(c00+3*c01,c10+3*c11)
                for c00 in range(3)
                for c01 in range(3)
                for c10 in range(3)
                for c11 in range(3)
                if (c00 + c10)%3 == i0
                and (c01 + c11)%3 == i1
                and 3*c11+c10 > 3*c01+c00]
        return mons
        
    def square_mons(mons):
        mons = [mon1+mon2 for mon1 in mons for mon2 in mons]
        return mons
    
    def get_monomials():
        monss = []
        for x in range(9):
            mons = index_to_mons(x)
            mons = square_mons(mons)
            monss.append(mons)
        return monss
    
    monss = get_monomials()
    ls = [len(mons) for mons in monss]
    
    def zero_list(length):
        return [0 for _ in range(length)]
    
    def get_rows(sample):
        R = sample[0]
        T = sample[1]
        RT = sample[2]
        rows = []
        for x in range(1,9):
            betweenzeros = sum([ls[it] for it in range(x-1)])
            postzeros = sum([ls[it] for it in range(x+1,9)])
                
            row = ([RT[x]*R[i]*R[j]*T[m]*T[n] for (i,j,m,n) in monss[0]]
                   + zero_list(betweenzeros)
                   + [-RT[0]*R[i]*R[j]*T[m]*T[n] for (i,j,m,n) in monss[x]]
                   + zero_list(postzeros)
                  )
    
            rows.append(row)
    
        return rows
        
    M = []
    for s in range(20):
        sample = get_sample()
        rows = get_rows(sample)
        M += rows
        
    M = matrix(Fp, M)

    K = M.right_kernel()
    
    b0 = K.basis()[0]
    
    B = Phi.codomain()
    
    hs = B._h
    ds = B._d
    
    return hs,ds,b0[0:17]


In [8]:
# ChatGPT method

############################################
# 2. Monomial index set
#    h_i h_j d_k d_l with symmetry
############################################

idxs = [
    (i1, j1, i2, j2)
    for i1 in range(5) for j1 in range(i1, 5)
    for i2 in range(4) for j2 in range(i2, 4)
]
ls = len(idxs)   # number of monomials

############################################
# 3. Monomial vector for one sample
############################################

def monomial_vector(hs, ds):
    """
    Returns m in F^ls where
    m[k] = h[i1]*h[j1]*d[i2]*d[j2]
    """
    return vector(F, [
        hs[i1] * hs[j1] * ds[i2] * ds[j2]
        for (i1, j1, i2, j2) in idxs
    ])

############################################
# 4. Reduced interpolation matrix
#    (Kronecker factorization)
############################################

def reduced_matrix(samples):
    """
    Builds the small interpolation matrix of size
    (16 * #samples) x 17
    """
    rows = []

    for sample in samples:
        hs, ds, cs = sample

        # one row per x = 1..16
        for x in range(1, 17):
            row = [F(0)] * 17
            row[x] = cs[x]
            row[0] = -cs[0]
            rows.append(row)

    return Matrix(F, rows)

############################################
# 5. Kernel lifting
############################################

def lift_kernel_vector(w, m):
    """
    Given:
      w in F^17   (kernel vector of reduced system)
      m in F^ls   (monomial vector)

    Returns:
      v in F^(17*ls) lifting w
    """
    v = []
    for x in range(17):
        v.extend([w[x] * mk for mk in m])
    return vector(F, v)

############################################
# 6. Example usage
############################################

# Collect samples
num_samples = 200
samples = [get_meta_sample() for _ in range(num_samples)]

# Build reduced matrix
R = reduced_matrix(samples)

print("Reduced matrix size:", R.nrows(), "x", R.ncols())

# Compute kernel
K = R.right_kernel()
print("Reduced kernel dimension:", K.dimension())

# Lift kernel vectors
m0 = monomial_vector(samples[0][0], samples[0][1])
lifted_kernel = [lift_kernel_vector(w, m0) for w in K.basis()]

print("Lifted kernel vectors:", len(lifted_kernel))
print("Ambient dimension:", 17 * ls)

Reduced matrix size: 3200 x 17
Reduced kernel dimension: 0
Lifted kernel vectors: 0
Ambient dimension: 2550


In [12]:
#ChatGPT method 2

############################################
# 1. Monomial index set
############################################

idxs = [
    (i1, j1, i2, j2)
    for i1 in range(5) for j1 in range(i1, 5)
    for i2 in range(4) for j2 in range(i2, 4)
]
ls = len(idxs)          # monomials per block
NBLOCKS = 17
DIM = NBLOCKS * ls      # total coefficients

############################################
# 2. Monomial evaluation vector
############################################

def monomial_vector(hs, ds):
    return vector(F, [
        hs[i1] * hs[j1] * ds[i2] * ds[j2]
        for (i1, j1, i2, j2) in idxs
    ])

############################################
# 3. Reduced kernel for ONE sample
############################################
# Encodes:  c_x * u_x = c_0 * u_0

def reduced_kernel(sample):
    hs, ds, cs = sample
    rows = []

    for x in range(1, NBLOCKS):
        row = [F(0)] * NBLOCKS
        row[x] = cs[x]
        row[0] = -cs[0]
        rows.append(row)

    R = Matrix(F, rows)
    return R.right_kernel()   # subspace of F^17

############################################
# 4. Lift reduced kernel to homogeneous constraints
############################################
# Unknowns live in F^(17*ls + 1)
# Last coordinate = projective scale t

def lifted_constraints(sample):
    hs, ds, cs = sample
    m = monomial_vector(hs, ds)

    K = reduced_kernel(sample)
    rows = []

    for w in K.basis():
        for x in range(NBLOCKS):
            row = [F(0)] * (DIM + 1)
            offset = x * ls

            # ⟨v_x , m⟩
            for k in range(ls):
                row[offset + k] = m[k]

            # − w_x * t
            row[DIM] = -w[x]

            rows.append(row)

    return rows

############################################
# 5. Incremental intersection
############################################

def interpolate_kernel(samples):
    constraints = []

    for i, sample in enumerate(samples):
        constraints.extend(lifted_constraints(sample))

        M = Matrix(F, constraints)
        K = M.right_kernel()

        print(f"After sample {i+1}: kernel dimension = {K.dimension()}")

        if K.dimension() == 0:
            break

    return K

############################################
# 6. Example / test harness
############################################

# Collect samples
num_samples = 200
samples = [get_meta_sample() for _ in range(num_samples)]

K = interpolate_kernel(samples)

print("\nFinal kernel dimension:", K.dimension())


AssertionError: 

In [ ]:
def get_inv(alphas):
    alpha0 = alphas[0]
    field = alpha0.parent()
    R.<q> = field[]
    [a0,a1,a2,a3,a4] = [alpha0] + [alpha/2 for alpha in alphas[1:]]
    f = field(1/4) * ((a0^4*a2^6 + 2*a0^4*a2^3*a4^3 + 32*a0*a1^3*a2^3*a4^3 + a0^4*a4^6)*q^6 + (-6*a0^4*a2^5*a3 + 24*a0^2*a1^2*a2^4*a4^2 - 6*a0^4*a2^2*a3*a4^3 - 96*a0*a1^3*a2^2*a3*a4^3 - 24*a0^2*a1^2*a2*a4^5)*q^5 + (15*a0^4*a2^4*a3^2 - 96*a0^2*a1^2*a2^3*a3*a4^2 + 6*a0^4*a2*a3^2*a4^3 + 96*a0*a1^3*a2*a3^2*a4^3 - 24*a0^3*a1*a2^2*a4^4 - 48*a1^4*a2^2*a4^4 + 24*a0^2*a1^2*a3*a4^5)*q^4 + (-20*a0^4*a2^3*a3^3 + 144*a0^2*a1^2*a2^2*a3^2*a4^2 - 2*a0^4*a2^3*a4^3 - 32*a0*a1^3*a2^3*a4^3 - 2*a0^4*a3^3*a4^3 - 32*a0*a1^3*a3^3*a4^3 + 48*a0^3*a1*a2*a3*a4^4 + 96*a1^4*a2*a3*a4^4 + 2*a0^4*a4^6 + 32*a0*a1^3*a4^6)*q^3 + (15*a0^4*a2^2*a3^4 - 96*a0^2*a1^2*a2*a3^3*a4^2 + 6*a0^4*a2^2*a3*a4^3 + 96*a0*a1^3*a2^2*a3*a4^3 - 24*a0^3*a1*a3^2*a4^4 - 48*a1^4*a3^2*a4^4 + 24*a0^2*a1^2*a2*a4^5)*q^2 + (-6*a0^4*a2*a3^5 + 24*a0^2*a1^2*a3^4*a4^2 - 6*a0^4*a2*a3^2*a4^3 - 96*a0*a1^3*a2*a3^2*a4^3 - 24*a0^2*a1^2*a3*a4^5)*q + a0^4*a3^6 + 2*a0^4*a3^3*a4^3 + 32*a0*a1^3*a3^3*a4^3 + a0^4*a4^6)
    C = HyperellipticCurve(f)
    return C.absolute_igusa_invariants_wamelen()

def get_inv2(alphas):
    alpha0 = alphas[0]
    field = alpha0.parent()
    R.<x> = field[]

    [a0,a1,a2,a3,a4] = [alpha/alpha0 for alpha in alphas]
    H3 = a4*(a2*x^2-a3*x-a1*a4)
    G3 = ( (a1^3*a4^3 + 3*a1*a2*a3*a4^4 + 2*a2^3*a4^3 + a2^3 + a3^3*a4^3)*x^3
          + (3*a1^2*a2*a4^5 - 3*a2^2*a3*a4^3 + 3*a1^2*a2*a4^2 - 3*a2^2*a3)*x^2
    + (-3*a1^2*a3*a4^5 + 3*a2*a3^2*a4^3 - 3*a1^2*a3*a4^2 + 3*a2*a3^2)*x
    + (-2*a1^3*a4^6 - a1^3*a4^3 + 3*a1*a2*a3*a4^4 + a2^3*a4^3 - a3^3)
         )
    lam3 = a1^3*a4^6 - 3*a1*a2*a3*a4^4 + a1^3*a4^3 - a2^3*a4^3 - a3^3*a4^3 - 3*a1*a2*a3*a4 - a2^3 - a3^3
    C = HyperellipticCurve(lam3*H3^3, G3)
    return C.absolute_igusa_invariants_wamelen()

map_list = Phi._maps

for k in range(1,9):
    map1 = map_list[3*k]
    map2 = map_list[3*(k+1)]
    
    cod = map1.codomain()
    dom = map2.domain()

    print(get_inv2(dom._h) == get_inv2(cod._h))

In [ ]:
def get_hyperelliptic(alphas):
    alpha0 = alphas[0]
    field = alpha0.parent()
    R.<x> = field[]

    [a0,a1,a2,a3,a4] = [alpha/alpha0 for alpha in alphas]
    H3 = a4*(a2*x^2-a3*x-a1*a4)
    G3 = ( (a1^3*a4^3 + 3*a1*a2*a3*a4^4 + 2*a2^3*a4^3 + a2^3 + a3^3*a4^3)*x^3
          + (3*a1^2*a2*a4^5 - 3*a2^2*a3*a4^3 + 3*a1^2*a2*a4^2 - 3*a2^2*a3)*x^2
    + (-3*a1^2*a3*a4^5 + 3*a2*a3^2*a4^3 - 3*a1^2*a3*a4^2 + 3*a2*a3^2)*x
    + (-2*a1^3*a4^6 - a1^3*a4^3 + 3*a1*a2*a3*a4^4 + a2^3*a4^3 - a3^3)
         )
    lam3 = a1^3*a4^6 - 3*a1*a2*a3*a4^4 + a1^3*a4^3 - a2^3*a4^3 - a3^3*a4^3 - 3*a1*a2*a3*a4 - a2^3 - a3^3
    C = HyperellipticCurve(lam3*H3^3, G3)
    return C

In [ ]:
get_hyperelliptic(dom._h).absolute_igusa_invariants_wamelen(), get_hyperelliptic(cod._h).absolute_igusa_invariants_wamelen()